# City-scale analysis: joins across layers

215k features share join keys (`client_seg_id`, `inspect_cell_id`) — combine
condition scores, distresses, and assets into city-scale insight.

In [ ]:
import cyvl

scene = cyvl.load_scene("somerville")
pav, dis = scene.pavements, scene.distresses

In [ ]:
# Worst streets by total distress area
worst = (
    dis.groupby("client_seg_id")["area_sq_ft"].sum().rename("distress_sqft")
    .to_frame().join(pav.set_index("client_seg_id")[["address_st", "score"]])
    .dropna().sort_values("distress_sqft", ascending=False)
)
worst.head(10)

In [ ]:
# Sign condition by MUTCD category
scene.signs.groupby(["category", "condition"]).size().unstack(fill_value=0)

In [ ]:
# For any asset, jump straight to photographic evidence: project its
# surveyed location into the nearest photo and circle it.
from matplotlib.patches import Circle

hydrant = scene.assets[scene.assets["asset_type"] == "HYDRANT"].iloc[9]
frame = scene.nearest_frame(hydrant.geometry.x, hydrant.geometry.y)
proj = frame.project(
    lonlat=[[hydrant.geometry.x, hydrant.geometry.y]], alt=frame.position[2] - 1.2
)

ax = frame.show()
radius_px = frame.camera.fx * 0.6 / proj.depth[0]   # ~0.6 m at the asset's depth
ax.add_patch(Circle(proj.pixels[0], radius_px, fill=False, color="red", linewidth=3))
ax.set_title(f"{hydrant['asset_type']} ({hydrant['condition']}) — {proj.depth[0]:.1f} m from camera")
